# The Oregon Medicaid experiment

This notebook reproduces the results from Chapter 4.1 'The Oregon Medicaid experiment' of the Approximate Maximum Perturbation (AMIP) paper using Python and the JAX library.

The link to the study paper is [here](https://pmc.ncbi.nlm.nih.gov/articles/PMC3535298/#S3).

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np

from functools import partial

# 1. Data Preparation


To reproduce the findings, I downloaded the Oregon Health Insurance Experiment dataset from the [original link](https://www.nber.org/research/data/oregon-health-insurance-experiment-data). The data preparation script was retained from the AMIP paper's [Github Repo](https://github.com/rgiordan/AMIPPaper).

In [2]:
# Load the CSV file into memory
df = pd.read_csv("/Users/yumin/Documents/Research/PML/Repository/amip-robustness-jax/data/OHIE_Public_Use_Files/OHIE_Data/data_for_analysis_R.csv")

print(f"The shape of the dataset before filtering: {df.shape}")

The shape of the dataset before filtering: (74922, 563)


/var/folders/mr/gyhq4nd5159cfdvmmd1l_d440000gn/T/ipykernel_13732/3469417847.py:2: DtypeWarning: Columns (0: female_cb) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/yumin/Documents/Research/PML/Repository/amip-robustness-jax/data/OHIE_Public_Use_Files/OHIE_Data/data_for_analysis_R.csv")


In [ ]:
def prepare_ohie_regression_data(orig_df, outcome_var):
    """
    Prepares the dataset for OHIE regression modeling for a specified outcome.
    """
    # 1. Filter for the 12-month survey respondents
    df = orig_df[orig_df['sample_12m_resp'] == 1].copy()
    
    # 2. Identify fixed effect columns (starting with "ddd")
    ddd_cols = [col for col in df.columns if col.startswith('ddd')]
    
    # 3. Define the core variables used in the model
    core_vars = [
        'treatment',             # Treatment indicator
        'household_id',          # Cluster ID for standard errors
        outcome_var,             # Target outcome variable
        'ohp_all_ever_survey',   # Endogenous instrument variable (for IV/LATE)
        'weight_12m'             # Final 12-month survey weights
    ]
    
    # 4. Subset the dataframe to only the variables needed for this specific model
    cols_to_keep = core_vars + ddd_cols
    regression_df = df[cols_to_keep].copy()
    
    # 5. Filter out missing values (equivalent to R's complete.cases)
    regression_df_clean = regression_df.dropna()
    
    # 6. Re-index the household_id cluster index (as performed in the R script)
    regression_df_clean.loc[:, 'household_id'] = (
        regression_df_clean['household_id'] - regression_df_clean['household_id'].min()
    )
    
    # Extract the prepared components for modeling
    y = regression_df_clean[outcome_var]
    X_treatment = regression_df_clean['treatment']
    X_instrument = regression_df_clean['ohp_all_ever_survey']
    X_fixed_effects = regression_df_clean[ddd_cols]
    weights = regression_df_clean['weight_12m']
    clusters = regression_df_clean['household_id']
    
    return regression_df_clean, y, X_treatment, X_instrument, X_fixed_effects, weights, clusters

In [4]:
# Define the 7 outcomes modelled on table 9 of the original paper
table_9_outcomes = [
    "health_genflip_bin_12m", 
    "health_notpoor_12m",
    "health_chgflip_bin_12m", 
    "notbaddays_tot_12m",
    "notbaddays_phys_12m", 
    "notbaddays_ment_12m",
    "nodep_screen_12m"
]

# 2. OLS

The AMIP paper evaluates both Intent-to-Treat (ITT) and Local Average Treatment Effect (LATE) models.

In [5]:
# @partial(jax.jit, static_argnames=['num_clusters'])
# def fit_wls(X, y, weights, clusters, num_clusters):
#     """
#     Helper function to fit a Weighted Least Squares (WLS) model
#     and return cluster-robust standard errors.
#     """
#     # Ensure weights are a column vector for broadcasting
#     sqrt_w = jnp.sqrt(weights).reshape(-1, 1)
#     X_weighted = X * sqrt_w

#     y_col = y.reshape(-1, 1) if y.ndim == 1 else y
#     y_weighted = y_col * sqrt_w

#     # 1. Calculate beta
#     beta, _, _, _ = jnp.linalg.lstsq(X_weighted, y_weighted)

#     # 2. Calculate residuals
#     residuals = y_weighted - jnp.dot(X_weighted, beta)

#     # 3. Calculate score vectors for each observation
#     obs_scores = X_weighted * residuals

#     # 4. Sum the scores by cluster to get s_g (Shape: num_clusters x P)
#     cluster_scores = jax.ops.segment_sum(obs_scores, clusters, num_segments=num_clusters)

#     # 5. Compute the B
#     B = jnp.dot(cluster_scores.T, cluster_scores)

#     # 6. Compute the XtwX
#     XTX_inv = jnp.linalg.inv(jnp.dot(X_weighted.T, X_weighted))

#     # 7. Finite-sample correction factor
#     N_effective = jnp.sum(weights > 0)

#     cluster_weights = jax.ops.segment_sum(weights, clusters, num_segments=num_clusters)
#     G_effective = jnp.sum(cluster_weights > 0) 
    
#     P = X.shape[1]

#     correction = (G_effective / (G_effective - 1)) * ((N_effective -1) / (N_effective - P))
    
#     cov_matrix = correction * jnp.dot(jnp.dot(XTX_inv, B), XTX_inv)

#     standard_errors = jnp.sqrt(jnp.diag(cov_matrix)).reshape(-1, 1)

#     return beta, standard_errors

In [6]:
@partial(jax.jit, static_argnames=['num_clusters'])
def fit_wls(X, y, weights, clusters, num_clusters):
    """
    Helper function to fit a Weighted Least Squares (WLS) model
    and return cluster-robust standard errors.
    """
    # Ensure weights are a column vector for broadcasting
    sqrt_w = jnp.sqrt(weights).reshape(-1, 1)
    X_weighted = X * sqrt_w

    y_col = y.reshape(-1, 1) if y.ndim == 1 else y
    y_weighted = y_col * sqrt_w

    # 1. Calculate beta
    beta, _, _, _ = jnp.linalg.lstsq(X_weighted, y_weighted)

    # 2. Calculate residuals
    residuals = y_weighted - jnp.dot(X_weighted, beta)

    # 3. Calculate score vectors for each observation
    obs_scores = X_weighted * residuals

    # 4. Sum the scores by cluster to get s_g (Shape: num_clusters x P)
    cluster_scores = jax.ops.segment_sum(obs_scores, clusters, num_segments=num_clusters)

    # 5. Compute the B
    B = jnp.dot(cluster_scores.T, cluster_scores)

    # 6. Compute the XtwX
    XTX_inv = jnp.linalg.inv(jnp.dot(X_weighted.T, X_weighted))

    # 7. CRV0 Variance-Covariance Matrix
    cov_matrix = jnp.dot(jnp.dot(XTX_inv, B), XTX_inv)

    standard_errors = jnp.sqrt(jnp.diag(cov_matrix)).reshape(-1, 1)

    return beta, standard_errors

## 2.1. ITT
**ITT (OLS):** $y_{ihj} = \beta_0 + \beta_1 \text{LOTTERY}_h + \beta_2 X_{ih} + \beta_3 V_{ih} + \epsilon_{ihj}$.


In [7]:
@partial(jax.jit, static_argnames=['num_clusters'])
def itt_model_weighted(y, lottery, X, V, weights, clusters, num_clusters):
    """
    Weighted Intent-to-Treat (ITT) Model using OLS.
    y = beta_0 + beta_1*LOTTERY + beta_2*X + beta_3*V + error
    """
    # 1. Create the intercept column (beta_0)
    intercept = jnp.ones((y.shape[0], 1))

    # 2. Build the design matrix: [Intercept, LOTTERY, X, V]
    # Ensure all 1D arrays are reshaped to 2D column vectors if necessary
    lottery_col = lottery.reshape(-1, 1) if lottery.ndim == 1 else lottery

    Z = jnp.column_stack([intercept, lottery_col, X, V])

    beta, se = fit_wls(Z, y, weights, clusters, num_clusters)

    return beta, se, Z

## 2.2. LATE
**LATE (IV):** Uses a two-stage least squares (2SLS)

* $y_{ihj} = \pi_0 + \pi_1 \text{INSURANCE}_{ih} + \pi_2 X_{ih} + \pi_3 V_{ih} + \upsilon_{ihj}$

* $\text{INSURANCE}_{ih} = \delta_0 + \delta_1 \text{LOTTERY}_h + \delta_2 X_{ih} + \delta_3 V_{ih} + \mu_{ihj}$

In [8]:
# @partial(jax.jit, static_argnames=['num_clusters'])
# def late_iv_model_weighted(y, insurance, lottery, X, V, weights, clusters, num_clusters):
#     """
#     Weighted Local Average Treatment Effect (LATE) Model using 2SLS.
#     Stage 1: INSURANCE = delta_0 + delta_1*LOTTERY + delta_2*X + delta_3*V
#     Stage 2: y = pi_0 + pi_1*INSURANCE_HAT + pi_2*X + pi_3*V
#     """
#     # 1. Create the intercept column
#     intercept = jnp.ones((y.shape[0],1))
#     lottery_col = lottery.reshape(-1, 1) if lottery.ndim == 1 else lottery
#     insurance_col = insurance.reshape(-1, 1) if insurance.ndim == 1 else insurance

#     # 2. Stage 1: Predict Insurance
#     Z_instruments = jnp.column_stack([intercept, lottery_col, X, V])

#     delta, _ = fit_wls(Z_instruments, insurance_col, weights, clusters, num_clusters)
#     insurance_hat = jnp.dot(Z_instruments, delta)

#     # 3. Stage 2: Estimate LATE coefficients
#     Z_fitted = jnp.column_stack([intercept, insurance_hat, X, V])
#     Z_true = jnp.column_stack([intercept, insurance_col, X, V])

#     # The SE is discarded because it is computed using the fitted residuals, not the true residuals.
#     pi, _ = fit_wls(Z_fitted, y, weights, clusters, num_clusters)

#     # 4. Compute True Residuals for Correct CRVE
#     sqrt_w = jnp.sqrt(weights).reshape(-1, 1)
#     Z_fitted_w = Z_fitted * sqrt_w
#     Z_true_w = Z_true * sqrt_w
#     y_w = (y.reshape(-1, 1) if y.ndim==1 else y) * sqrt_w

#     true_residuals = y_w - jnp.dot(Z_true_w, pi)

#     # 5. Construct the Correct Sandwich Matrix; The scores must be generated using the fitted design matrix but the True rediausls
#     obs_scores = Z_fitted_w * true_residuals

#     cluster_scores = jax.ops.segment_sum(obs_scores, clusters, num_segments=num_clusters)
#     B = jnp.dot(cluster_scores.T, cluster_scores)

#     XTX_inv = jnp.linalg.inv(jnp.dot(Z_fitted_w.T, Z_fitted_w))

#     # Finite-sample correction factor
#     N_effective = jnp.sum(weights > 0)
#     cluster_weights = jax.ops.segment_sum(weights, clusters, num_segments=num_clusters)
#     G_effective = jnp.sum(cluster_weights > 0)
#     P = Z_fitted.shape[1]

#     correction = (G_effective / (G_effective - 1)) * ((N_effective - 1) / (N_effective - P))

#     cov_matrix = correction * jnp.dot(jnp.dot(XTX_inv, B), XTX_inv)
#     se2 = jnp.sqrt(jnp.diag(cov_matrix)).reshape(-1, 1)
    

#     return delta, pi, se2, Z_fitted, Z_true

In [9]:
@partial(jax.jit, static_argnames=['num_clusters'])
def late_iv_model_weighted(y, insurance, lottery, X, V, weights, clusters, num_clusters):
    """
    Weighted Local Average Treatment Effect (LATE) Model using 2SLS.
    Stage 1: INSURANCE = delta_0 + delta_1*LOTTERY + delta_2*X + delta_3*V
    Stage 2: y = pi_0 + pi_1*INSURANCE_HAT + pi_2*X + pi_3*V
    """
    # 1. Create the intercept column
    intercept = jnp.ones((y.shape[0],1))
    lottery_col = lottery.reshape(-1, 1) if lottery.ndim == 1 else lottery
    insurance_col = insurance.reshape(-1, 1) if insurance.ndim == 1 else insurance

    # 2. Stage 1: Predict Insurance
    Z_instruments = jnp.column_stack([intercept, lottery_col, X, V])

    delta, _ = fit_wls(Z_instruments, insurance_col, weights, clusters, num_clusters)
    insurance_hat = jnp.dot(Z_instruments, delta)

    # 3. Stage 2: Estimate LATE coefficients
    Z_fitted = jnp.column_stack([intercept, insurance_hat, X, V])
    Z_true = jnp.column_stack([intercept, insurance_col, X, V])

    # The SE is discarded because it is computed using the fitted residuals, not the true residuals.
    pi, _ = fit_wls(Z_fitted, y, weights, clusters, num_clusters)

    # 4. Compute True Residuals for Correct CRVE
    sqrt_w = jnp.sqrt(weights).reshape(-1, 1)
    Z_fitted_w = Z_fitted * sqrt_w
    Z_true_w = Z_true * sqrt_w
    y_w = (y.reshape(-1, 1) if y.ndim==1 else y) * sqrt_w

    true_residuals = y_w - jnp.dot(Z_true_w, pi)

    # 5. Construct the Correct Sandwich Matrix; The scores must be generated using the fitted design matrix but the True rediausls
    obs_scores = Z_fitted_w * true_residuals

    cluster_scores = jax.ops.segment_sum(obs_scores, clusters, num_segments=num_clusters)
    B = jnp.dot(cluster_scores.T, cluster_scores)

    XTX_inv = jnp.linalg.inv(jnp.dot(Z_fitted_w.T, Z_fitted_w))

    # CRV0 Variance-Covariance Matrix
    cov_matrix = jnp.dot(jnp.dot(XTX_inv, B), XTX_inv)
    se2 = jnp.sqrt(jnp.diag(cov_matrix)).reshape(-1, 1)
    

    return delta, pi, se2, Z_fitted, Z_true

## 2.3. Testing

In [10]:
for outcome in table_9_outcomes:

    # 1. Prepare data for the current outcome
    clean_df, y, X_treat, X_inst, X_fe, w, clusters = prepare_ohie_regression_data(
        orig_df=df, 
        outcome_var=outcome
    )

    # 2. Convert the pandas Series and DataFrames to JAX arrays
    y_jax = jnp.array(y.values)
    lottery_jax = jnp.array(X_treat.values)
    insurance_jax = jnp.array(X_inst.values)
    X_jax = jnp.array(X_fe.values)
    weights_jax = jnp.array(w.values)
    clusters_jax = jnp.array(clusters.values)
    num_clusters = int(jnp.max(clusters_jax)) + 1

    # 3. Handle the V matrix (Optional covariates)
    # The R script for Table 9 only utilizes the 'ddd' fixed effects (which we mapped to X).
    # To satisfy the JAX function signature without affecting the math, we pass a zero-column array for V.
    V_jax = jnp.empty((y_jax.shape[0], 0))

    # 4. Execute the ITT (OLS) Model
    itt_betas, itt_se, itt_Z = itt_model_weighted(
        y=y_jax, 
        lottery=lottery_jax, 
        X=X_jax, 
        V=V_jax,
        weights=weights_jax,
        clusters=clusters_jax,
        num_clusters=num_clusters
    )

    # 5. Execute the LATE (2SLS) Model
    late_deltas, late_pis, late_se, late_Z_fitted, late_Z_true = late_iv_model_weighted(
        y=y_jax, 
        insurance=insurance_jax, 
        lottery=lottery_jax, 
        X=X_jax, 
        V=V_jax,
        weights=weights_jax,
        clusters=clusters_jax,
        num_clusters=num_clusters
    )

    # 6. Extract the coefficients of interest
    # Index 0 is the intercept, Index 1 is the treatment/insurance effect
    itt_estimate = itt_betas[1]
    late_estimate = late_pis[1]

    print(f"Outcome: {outcome}")
    print(f"ITT Estimate: {itt_estimate[0]:.3f} | Standard Error: {itt_se[1][0]:.3f}")
    print(f"LATE Estimate: {late_estimate[0]:.3f} | Standard Error: {late_se[1][0]:.3f}")
    

Outcome: health_genflip_bin_12m
ITT Estimate: 0.039 | Standard Error: 0.008
LATE Estimate: 0.133 | Standard Error: 0.026
Outcome: health_notpoor_12m
ITT Estimate: 0.029 | Standard Error: 0.005
LATE Estimate: 0.099 | Standard Error: 0.018
Outcome: health_chgflip_bin_12m
ITT Estimate: 0.033 | Standard Error: 0.007
LATE Estimate: 0.113 | Standard Error: 0.023
Outcome: notbaddays_tot_12m
ITT Estimate: 0.381 | Standard Error: 0.162
LATE Estimate: 1.317 | Standard Error: 0.563
Outcome: notbaddays_phys_12m
ITT Estimate: 0.459 | Standard Error: 0.175
LATE Estimate: 1.585 | Standard Error: 0.606
Outcome: notbaddays_ment_12m
ITT Estimate: 0.603 | Standard Error: 0.184
LATE Estimate: 2.082 | Standard Error: 0.640
Outcome: nodep_screen_12m
ITT Estimate: 0.023 | Standard Error: 0.007
LATE Estimate: 0.078 | Standard Error: 0.025


We have successfully reproduced the OLS results from the original paper. Now we implement the AMIP.

# 3. ITT: AMIP JAX Implementation

In this section, I detail the mathematical implementation of AMIP.

We start with the core linear approximation of the AMIP framework. The authors want to approximate what happens to our conclusion, $\phi$, when we tweak the weights, $\vec{w}$, of our data points.

The Taylor series approximation is given as:


$$\phi(\hat{\theta}(\vec{w}), \vec{w}) \approx \phi^{lin}(\vec{w}) := \hat{\phi} + \sum_{n=1}^N (w_n - 1)\psi_n$$

Here, the critical missing puzzle piece is $\psi_n$. The paper defines $\psi_n$ as the rate of change (the derivative) of our conclusion with respect to the weight of the $n$-th data point, evaluated when all weights are equal to 1. Mathematically, this is written as:


$$\psi_n := \frac{\partial\phi(\hat{\theta}(\vec{w}), \vec{w})}{\partial w_n}\bigg\vert{}_{\vec{w}=\vec{1}}$$

Define the following as the *influence score* of data point n for $\psi$ at $\vec{w}$:

$$\frac{\partial\phi(\hat{\theta}(\vec{w}), \vec{w})}{\partial w_n}\bigg\vert{}_{\vec{w}}$$

To compute the influence score, we first apply the chain rule (Equation 7):

$$\frac{\partial\phi(\hat{\theta}(\vec{w}),\vec{w})}{\partial w_n}\bigg\vert{}_{\hat{\theta}(\vec{w}), \vec{w}} = \frac{\partial\phi(\theta,\vec{w})}{\partial\theta^T}\bigg\vert{}_{\hat{\theta}(\vec{w}),\vec{w}} \frac{\partial\hat{\theta}(\vec{w})}{\partial w_n}\bigg\vert{}_{\vec{w}} + \frac{\partial\phi(\theta,\vec{w})}{\partial w_n}\bigg\vert{}_{\hat{\theta}(\vec{w}),\vec{w}}$$

In order to solve for the second term on the RHS, we use the implicit function theorem which gives us: 

$$\frac{d\hat{\theta}(\vec{w})}{d\vec{w}^T}\bigg\vert{}_{\vec{w}} = -\left(\sum_{n=1}^N w_n \frac{\partial G(\theta,d_n)}{\partial\theta^T}\bigg\vert{}_{\hat{\theta}(\vec{w})}\right)^{-1} \left(G(\hat{\theta}(\vec{w}),d_1), ..., G(\hat{\theta}(\vec{w}),d_N)\right)$$

In [11]:
@jax.jit
def G(betas, x_n, y_n, survey_weight_n):
    """The gradient of the squared error for one person."""
    residual = y_n - jnp.dot(x_n, betas)
    gradient = -survey_weight_n * residual * x_n
    return gradient

In [12]:
@jax.jit
def compute_itt_influence_scores(beta_hat, X, y, weights, param_index):
    """
    Compute the influence score for every data point.

    Args:
        beta_hat: Array of shape (P,) representing the baseline OLS estimates.
        X: Matrix of shape (N, P) containing covariates.
        y: Array of shape (N,) containing the outcome variable.
        weights: Matrix of shape (N,) containing survey weights
        param_index: Integer index of the parameter we care about.

    Returns:
        psi: Array of shape (N,) containing the influence score for each observation.
    """
    N, P = X.shape

    # 1. Compute the Hessian (Derivative of G)
    hessian_G = jax.jacfwd(G, argnums=0)

    # Vectorize the functions so they can evaluate the whole dataset at once.
    G_vmap = jax.vmap(G, in_axes=(None, 0, 0, 0))
    hessian_G_vmap = jax.vmap(hessian_G, in_axes=(None, 0, 0, 0))

    # Evaluate G and the Hessian for every single data point at the baseline beta_hat
    G_vals = G_vmap(beta_hat, X, y, weights)         # Shape: (N, P)
    H_vals = hessian_G_vmap(beta_hat, X, y, weights) # Shape: (N, P, P)

    # 2. Aggregate and Invert the Hessian
    H_sum = jnp.sum(H_vals, axis=0)         # Shape: (P, P)
    H_inv = jnp.linalg.inv(H_sum)           # Shape: (P, P)

    # 3. Calculate the influence score
    # Define the derivative of our quantity of interest \phi(\beta)
    # Because \phi(\beta) = \beta_1, its derivative is a vector of zeros with a 1 at param_index
    d_phi = jnp.zeros(P).at[param_index].set(1.0)

    # Calculate the right side of the chain rule equation: H_inv @ d_phi
    # Shape will be (P,)
    chain_rule_vector = jnp.dot(H_inv.T, d_phi)

    # Multiply by G(\beta, d_n) for every person to get their specific \psi_n
    # The negative sign applies the chain rule formula from the AMIP implicit function theorem
    # Shape will be (N,)
    psi = -jnp.dot(G_vals, chain_rule_vector)

    return psi

Above, we use associative identity of matrix multiplication to compute the influence score.

Justification:

$$\psi^T = \frac{\partial\phi}{\partial\theta^T} \left( -H^{-1} G \right)$$
(where $H^{-1}$ is the inverted Hessian and $G$ is the matrix of gradients)

If we followed the strict left-to-right reading of the math, we would first multiply $-H^{-1}$ by $G$.

* $H^{-1}$ is a matrix of shape $P \times P$ (where $P$ is the number of parameters).
* $G$ is a massive matrix of shape $P \times N$ (where $N$ is the number of data points, which could be millions).

Multiplying a $P \times P$ matrix by a $P \times N$ matrix requires $O(NP^2)$ operations. This is highly inefficient and consumes a massive amount of memory just to create an intermediate matrix.

Instead, we can shift the parentheses to group the first two terms together instead:

$$\psi^T = - \left( \frac{\partial\phi}{\partial\theta^T} H^{-1} \right) G$$

Now, the dimensions of this new grouping as follows:

* $\frac{\partial\phi}{\partial\theta^T}$ is a vector of shape $1 \times P$.
* $H^{-1}$ is a matrix of shape $P \times P$.

Multiplying these together first takes almost zero time because $P$ is usually a very small number. It collapses immediately into a single vector of shape $1 \times P$. We then multiply that single tiny vector by the massive $G$ matrix, which only requires $O(NP)$ operations—a dramatic computational shortcut.


Finally, notice in our code, `G_vals` comes out with the shape $N \times P$ (rows of data points, columns of parameters), rather than $P \times N$.

To make the math perfectly align with JAX's $N \times P$ shape, we simply transpose the entire associative equation to solve for a column vector $\psi$:

$$\psi = - G^T \left( H^{-1} \right)^T \left( \frac{\partial\phi}{\partial\theta^T} \right)^T$$

This final mathematical transformation perfectly maps to the last two lines of the JAX code:

* **`chain_rule_vector = jnp.dot(H_inv.T, d_phi)`**
Computes the $\left( H^{-1} \right)^T \left( \frac{\partial\phi}{\partial\theta^T} \right)^T$ portion first.
* **`psi = -jnp.dot(G_vals, chain_rule_vector)`**
Multiplies the $N \times P$ transposed $G$ matrix by the newly created vector.

In [13]:
def find_amip_subset(psi_scores, beta_estimate, standard_error, target_type):
    """
    Finds the indices of the data points to drop to achieve a specific target.

    target_type can be: 'sign_change', 'significance_change', 'sign_reversal'.
    """
    N = len(psi_scores)

    # 1. Define the magnitude of change (Delta) required for our specific goal
    abs_beta = jnp.abs(beta_estimate)
    margin_of_error = 1.96 * standard_error

    if target_type == 'sign_change':
        delta = abs_beta
    elif target_type == 'significance_change':
        delta = abs_beta - margin_of_error
        if delta <= 0:
            print("Estimate is already statistically insignificant!")
            return jnp.array([]), 0.0
    elif target_type == 'sign_reversal':
        delta = abs_beta + margin_of_error
    else:
        raise ValueError("Invalid target_type")

    # 2. Sort the influence scores
    if beta_estimate > 0:
        sort_indices = jnp.argsort(-psi_scores)
        direction_multiplier = -1
    else:
        sort_indices = jnp.argsort(psi_scores)
        direction_multiplier = 1
    
    sorted_psi = psi_scores[sort_indices]

    # 3. Iteratively accumulate the change
    cumulative_magnitude_change = jnp.cumsum(-sorted_psi * direction_multiplier)

    threshold_met_mask = cumulative_magnitude_change >= delta

    if not jnp.any(threshold_met_mask):
        print(f"Target '{target_type}' cannot be reached even if we drop all data.")
        return None, None
    
    M = jnp.argmax(threshold_met_mask) + 1
    alpha = M / N

    indices_to_drop = sort_indices[:M]

    return indices_to_drop, alpha

In [14]:
def run_oregon_amip_itt_analysis(y, lottery, X, V, weights, clusters, num_clusters, param_index=1):
    """
    Executes the AMIP pipeline for the Oregon ITT model using JAX WLS.
    param_index=1 maps to the LOTTERY column.
    """
    N = y.shape[0]

    # Step 0: Baseline Estimation
    print("--- Baseline Oregon ITT Model ---")
    baseline_beta, baseline_se, Z_matrix = itt_model_weighted(y, lottery, X, V, weights, clusters, num_clusters)

    # Extract scalars from the returned column vectors
    target_beta = baseline_beta[param_index][0]
    target_se = baseline_se[param_index][0]
    print(f"Lottery Estimate: {target_beta:.3f} | Standard Error: {target_se:.3f}\n")

    # Step 1: Calculate Influence Scores
    psi_scores = compute_itt_influence_scores(
        beta_hat = baseline_beta.flatten(),
        X=Z_matrix,
        y=y.flatten(),
        weights=weights.flatten(),
        param_index=param_index
    )

    cluster_psi = jax.ops.segment_sum(psi_scores, clusters, num_segments=num_clusters)

    targets = ['sign_change', 'significance_change', 'sign_reversal']

    for target in targets:
        print(f"--- Target: {target.upper()} ---")

        # Steps 2 & 3: Find the subset size (M)
        drop_indices, alpha = find_amip_subset(psi_scores, target_beta, target_se, target)

        if alpha is None or len(drop_indices) == 0:
            continue

        print(f"Points to drop (M): {len(drop_indices)}")
        print(f"Proportion (alpha): {alpha*100:.2f}%")

        # Step 4: The Validation Refit (Using WLS function)
        #refit_weights = jnp.ones(N).at[drop_indices].set(0.0)
        refit_weights = weights.at[drop_indices].set(0.0)

        # Re-run the ITT model with the new weights
        refit_beta, refit_se, _ = itt_model_weighted(y, lottery, X, V, refit_weights, clusters, num_clusters)

        actual_new_estimate = refit_beta[param_index][0]
        actual_new_se = refit_se[param_index][0]

        print(f"Validation Refit Estimate: {actual_new_estimate:.3f} | Refit SE: {actual_new_se:.3f}\n")

In [15]:
for outcome in table_9_outcomes:

    # 1. Prepare data for the current outcome
    clean_df, y, X_treat, X_inst, X_fe, w, clusters = prepare_ohie_regression_data(
        orig_df=df, 
        outcome_var=outcome
    )

    # 2. Convert the pandas Series and DataFrames to JAX arrays
    y_jax = jnp.array(y.values)
    lottery_jax = jnp.array(X_treat.values)
    insurance_jax = jnp.array(X_inst.values)
    X_jax = jnp.array(X_fe.values)
    weights_jax = jnp.array(w.values)
    clusters_jax = jnp.array(clusters.values)
    num_clusters = int(jnp.max(clusters_jax)) + 1

    # 3. Handle the V matrix (Optional covariates)
    # The R script for Table 9 only utilizes the 'ddd' fixed effects (which we mapped to X).
    # To satisfy the JAX function signature without affecting the math, we pass a zero-column array for V.
    V_jax = jnp.empty((y_jax.shape[0], 0))

    # 4. Execute the ITT (OLS) Model
    run_oregon_amip_itt_analysis(
        y = y_jax,
        lottery = lottery_jax,
        X = X_jax,
        V = V_jax,
        weights = weights_jax,
        clusters = clusters_jax,
        num_clusters = num_clusters,
        param_index = 1
    )
    

--- Baseline Oregon ITT Model ---
Lottery Estimate: 0.039 | Standard Error: 0.008

--- Target: SIGN_CHANGE ---
Points to drop (M): 286
Proportion (alpha): 1.22%
Validation Refit Estimate: -0.004 | Refit SE: 0.008

--- Target: SIGNIFICANCE_CHANGE ---
Points to drop (M): 158
Proportion (alpha): 0.68%
Validation Refit Estimate: 0.013 | Refit SE: 0.008

--- Target: SIGN_REVERSAL ---
Points to drop (M): 430
Proportion (alpha): 1.84%
Validation Refit Estimate: -0.022 | Refit SE: 0.008

--- Baseline Oregon ITT Model ---
Lottery Estimate: 0.029 | Standard Error: 0.005

--- Target: SIGN_CHANGE ---
Points to drop (M): 155
Proportion (alpha): 0.66%
Validation Refit Estimate: -0.001 | Refit SE: 0.005

--- Target: SIGNIFICANCE_CHANGE ---
Points to drop (M): 90
Proportion (alpha): 0.39%
Validation Refit Estimate: 0.009 | Refit SE: 0.005

--- Target: SIGN_REVERSAL ---
Points to drop (M): 246
Proportion (alpha): 1.05%
Validation Refit Estimate: -0.012 | Refit SE: 0.004

--- Baseline Oregon ITT Model -

# 4. LATE: AMIP JAX Implementation

In [16]:
@jax.jit
def G_late(pi, z_inst_n, z_true_n, y_n, survey_weight_n):
    """The estimating equation for exactly identified IV/2SLS."""
    # The residual is calculated using the TRUE endogenous variable (Insurance)
    true_residual = y_n - jnp.dot(z_true_n, pi)
    
    # The score is multiplied by the INSTRUMENT variable (Lottery)
    gradient = -survey_weight_n * true_residual * z_inst_n
    
    return gradient

In [17]:
@jax.jit
def compute_late_influence_scores(pi_hat, Z_inst, Z_true, y, weights, param_index):
    N, P = Z_inst.shape

    # Differentiate G_late with respect to pi (argnums=0)
    hessian_G = jax.jacfwd(G_late, argnums=0)

    # Vectorize across the batch dimension
    G_vmap = jax.vmap(G_late, in_axes=(None, 0, 0, 0, 0))
    hessian_G_vmap = jax.vmap(hessian_G, in_axes=(None, 0, 0, 0, 0))

    # Evaluate at baseline pi_hat
    G_vals = G_vmap(pi_hat, Z_inst, Z_true, y, weights)         
    H_vals = hessian_G_vmap(pi_hat, Z_inst, Z_true, y, weights) 

    # Aggregate and invert the Hessian
    H_sum = jnp.sum(H_vals, axis=0)         
    H_inv = jnp.linalg.inv(H_sum)           

    d_phi = jnp.zeros(P).at[param_index].set(1.0)
    chain_rule_vector = jnp.dot(H_inv.T, d_phi)

    # Calculate final influence scores
    psi = -jnp.dot(G_vals, chain_rule_vector)

    return psi

In [18]:
def run_oregon_amip_late_analysis(y, insurance, lottery, X, V, weights, clusters, num_clusters, param_index=1):
    """
    Executes the AMIP pipeline for the Oregon LATE model using JAX WLS.
    param_index=1 maps to the LOTTERY column.
    """
    N = y.shape[0]

    # Step 0: Baseline Estimation
    print("--- Baseline Oregon LATE Model ---")
    baseline_delta, baseline_pi, baseline_se, Z_matrix_fitted, Z_matrix_true = late_iv_model_weighted(y, insurance, lottery, X, V, weights, clusters, num_clusters)

    # Extract scalars from the returned column vectors
    target_beta = baseline_pi[param_index][0]
    target_se = baseline_se[param_index][0]
    print(f"Lottery Estimate: {target_beta:.3f} | Standard Error: {target_se:.3f}\n")

    # Step 1: Calculate Influence Scores
    psi_scores = compute_late_influence_scores(
        pi_hat = baseline_pi.flatten(),
        Z_inst=Z_matrix_fitted,
        Z_true=Z_matrix_true,
        y=y.flatten(),
        weights=weights.flatten(),
        param_index=param_index
    )

    cluster_psi = jax.ops.segment_sum(psi_scores, clusters, num_segments=num_clusters)

    targets = ['sign_change', 'significance_change', 'sign_reversal']

    for target in targets:
        print(f"--- Target: {target.upper()} ---")

        # Steps 2 & 3: Find the subset size (M)
        drop_indices, alpha = find_amip_subset(psi_scores, target_beta, target_se, target)

        if alpha is None or len(drop_indices) == 0:
            continue

        print(f"Points to drop (M): {len(drop_indices)}")
        print(f"Proportion (alpha): {alpha*100:.2f}%")

        # Step 4: The Validation Refit (Using WLS function)
        #refit_weights = jnp.ones(N).at[drop_indices].set(0.0)
        refit_weights = weights.at[drop_indices].set(0.0)

        # Re-run the LATE model with the new weights
        _, refit_pi, refit_se, _, _ = late_iv_model_weighted(y, insurance, lottery, X, V, refit_weights, clusters, num_clusters)

        actual_new_estimate = refit_pi[param_index][0]
        actual_new_se = refit_se[param_index][0]

        print(f"Validation Refit Estimate: {actual_new_estimate:.3f} | Refit SE: {actual_new_se:.3f}\n")

In [19]:
for outcome in table_9_outcomes:

    # 1. Prepare data for the current outcome
    clean_df, y, X_treat, X_inst, X_fe, w, clusters = prepare_ohie_regression_data(
        orig_df=df, 
        outcome_var=outcome
    )

    # 2. Convert the pandas Series and DataFrames to JAX arrays
    y_jax = jnp.array(y.values)
    lottery_jax = jnp.array(X_treat.values)
    insurance_jax = jnp.array(X_inst.values)
    X_jax = jnp.array(X_fe.values)
    weights_jax = jnp.array(w.values)
    clusters_jax = jnp.array(clusters.values)
    num_clusters = int(jnp.max(clusters_jax)) + 1

    # 3. Handle the V matrix (Optional covariates)
    # The R script for Table 9 only utilizes the 'ddd' fixed effects (which we mapped to X).
    # To satisfy the JAX function signature without affecting the math, we pass a zero-column array for V.
    V_jax = jnp.empty((y_jax.shape[0], 0))

    # 4. Execute the ITT (OLS) Model
    run_oregon_amip_late_analysis(
        y = y_jax,
        insurance = insurance_jax,
        lottery = lottery_jax,
        X = X_jax,
        V = V_jax,
        weights = weights_jax,
        clusters = clusters_jax,
        num_clusters = num_clusters,
        param_index = 1
    )
    

--- Baseline Oregon LATE Model ---
Lottery Estimate: 0.133 | Standard Error: 0.026

--- Target: SIGN_CHANGE ---
Points to drop (M): 275
Proportion (alpha): 1.18%
Validation Refit Estimate: -0.006 | Refit SE: 0.025

--- Target: SIGNIFICANCE_CHANGE ---
Points to drop (M): 155
Proportion (alpha): 0.66%
Validation Refit Estimate: 0.050 | Refit SE: 0.025

--- Target: SIGN_REVERSAL ---
Points to drop (M): 405
Proportion (alpha): 1.73%
Validation Refit Estimate: -0.052 | Refit SE: 0.024

--- Baseline Oregon LATE Model ---
Lottery Estimate: 0.099 | Standard Error: 0.018

--- Target: SIGN_CHANGE ---
Points to drop (M): 155
Proportion (alpha): 0.66%
Validation Refit Estimate: -0.003 | Refit SE: 0.015

--- Target: SIGNIFICANCE_CHANGE ---
Points to drop (M): 89
Proportion (alpha): 0.38%
Validation Refit Estimate: 0.033 | Refit SE: 0.016

--- Target: SIGN_REVERSAL ---
Points to drop (M): 247
Proportion (alpha): 1.06%
Validation Refit Estimate: -0.040 | Refit SE: 0.015

--- Baseline Oregon LATE Mode